# Model registry: MLFlow

Example on how to use the MLFlow model during the training of a model

## 1. Install MLFlow

In [ ]:
# !pip install -r requirements.txt
# !pip freeze | grep mlflow


## 2. Start the MLFlow server.

To do this, run the following command in a terminal:

mlflow ui --host 0.0.0.0 --port 5001

## 3. Create a new experiment

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("mlflow-model-training-iris-2")

client = MlflowClient()

## 4. Load and register the dataset

We use the Iris dataset and register it with MLflow so it appears attached to every run and model version in the UI.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from mlflow.models import infer_signature

# Load iris dataset
iris = datasets.load_iris()
X, y = iris.data, iris.target
feature_names = list(iris.feature_names)
target_names = list(iris.target_names)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Register the dataset — it will appear in the MLflow UI linked to runs and model versions
dataset = mlflow.data.from_numpy(X, targets=y, name="iris", source="sklearn.datasets.load_iris")

print(f"Dataset shape: {X.shape} | Classes: {target_names}")

## 5. Train a single model and log everything

This run demonstrates the core MLflow logging primitives: hyperparameters, metrics, tags, the dataset, a confusion matrix, and feature importances.

In [ ]:
import datetime

hyperparameters = {"n_estimators": 10, "max_depth": 5}

with mlflow.start_run(run_name=f"single-run-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"):
    mlflow.log_input(dataset, context="training")
    mlflow.set_tags({"model_type": "random_forest", "dataset": "iris"})
    mlflow.log_params(hyperparameters)

    model = RandomForestClassifier(**hyperparameters, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))

    # Confusion matrix
    fig, ax = plt.subplots()
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=target_names, ax=ax)
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    # Feature importance
    fig, ax = plt.subplots()
    ax.barh(feature_names, model.feature_importances_)
    ax.set_xlabel("Importance")
    ax.set_title("Feature Importance")
    mlflow.log_figure(fig, "feature_importance.png")
    plt.close(fig)

    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(model, "model", signature=signature)

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

## 6. Hyperparameter search with Optuna (nested runs)

Each trial is logged as a **child run** nested inside a parent sweep run. This keeps the UI clean: you see one `optuna-sweep` entry that groups all 20 trials underneath it.

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 10, 100),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_float("max_features", 0.1, 1.0),
    }
    with mlflow.start_run(nested=True, run_name=f"trial-{trial.number}"):
        mlflow.log_params(params)
        score = cross_val_score(RandomForestClassifier(**params, random_state=42), X_train, y_train, cv=5).mean()
        mlflow.log_metric("mean_cv_score", score)
    return score

with mlflow.start_run(run_name="optuna-sweep"):
    mlflow.log_input(dataset, context="training")
    mlflow.set_tags({"model_type": "random_forest", "dataset": "iris"})

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)

    best_params = study.best_params
    mlflow.log_metric("best_cv_score", study.best_value)

print(f"Best CV score: {study.best_value:.4f}")
print(f"Best params:   {best_params}")

## 7. Register the best model

Train on the full training set with the best hyperparameters, log all artifacts, and register the model in MLflow's Model Registry. We then add a description and assign the `champion` alias so it can be loaded by name instead of version number.

In [ ]:
MODEL_NAME = "random_forest_iris"

with mlflow.start_run(run_name="final-model"):
    mlflow.log_input(dataset, context="training")
    mlflow.set_tags({"model_type": "random_forest", "dataset": "iris"})
    mlflow.log_params(best_params)

    final_model = RandomForestClassifier(**best_params, random_state=42)
    final_model.fit(X_train, y_train)
    y_pred_final = final_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred_final)
    mlflow.log_metric("accuracy", accuracy)

    fig, ax = plt.subplots()
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_final, display_labels=target_names, ax=ax)
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    fig, ax = plt.subplots()
    ax.barh(feature_names, final_model.feature_importances_)
    ax.set_xlabel("Importance")
    ax.set_title("Feature Importance")
    mlflow.log_figure(fig, "feature_importance.png")
    plt.close(fig)

    signature = infer_signature(X_train, final_model.predict(X_train))
    mlflow.sklearn.log_model(final_model, "model", registered_model_name=MODEL_NAME, signature=signature)

    print(f"Accuracy: {accuracy:.4f} | Registered as: {MODEL_NAME}")

# Add a human-readable description to the registered model
client.update_registered_model(
    MODEL_NAME,
    description="Random Forest trained on Iris with Optuna hyperparameter search."
)

# Assign the 'champion' alias to the latest registered version
latest_version = client.get_registered_model(MODEL_NAME).latest_versions[0].version
client.set_registered_model_alias(MODEL_NAME, "champion", latest_version)
print(f"Alias 'champion' → version {latest_version}")

## 8. Evaluate the model

`mlflow.evaluate()` automatically computes a full suite of classification metrics and logs them — no manual metric computation needed.

In [ ]:
import pandas as pd

eval_df = pd.DataFrame(X_test, columns=feature_names)
eval_df["target"] = y_test

with mlflow.start_run(run_name="model-evaluation"):
    mlflow.evaluate(
        model=f"models:/{MODEL_NAME}@champion",
        data=eval_df,
        targets="target",
        model_type="classifier",
        evaluators="default",
    )

## 9. Load a model and run predictions

You can load a model by version number or by a named alias. Aliases are better in production — promoting a new version requires no code changes.

In [ ]:
import mlflow.pyfunc

# Load by version number
model_v1 = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/1")
print("By version:", model_v1.predict(X_test[:3]))

# Load by alias — no hardcoded version number
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
print("By alias:  ", champion.predict(X_test[:3]))

## 10. Explore with MlflowClient

`MlflowClient` gives you a programmatic API to inspect experiments, runs, and registered models — the same data you see in the UI.

In [ ]:
# List all experiments
print("=== Experiments ===")
for exp in client.search_experiments():
    print(f"  [{exp.experiment_id}] {exp.name}")

# Top 5 runs in the current experiment sorted by accuracy
exp = client.get_experiment_by_name("mlflow-model-training-iris")
runs = client.search_runs(exp.experiment_id, order_by=["metrics.accuracy DESC"], max_results=5)
print("\n=== Top 5 runs by accuracy ===")
for run in runs:
    acc = run.data.metrics.get("accuracy", "—")
    print(f"  {run.info.run_name}: accuracy={acc}")

# Registered model versions and their aliases
print(f"\n=== Versions of '{MODEL_NAME}' ===")
for mv in client.search_model_versions(f"name='{MODEL_NAME}'"):
    print(f"  version={mv.version}  aliases={mv.aliases}")

## 11. Cleanup

Run this cell to delete all runs and the registered model so you can start fresh. **Irreversible** — use with care.

In [ ]:
exp = client.get_experiment_by_name("mlflow-model-training-iris")
if exp:
    for run in client.search_runs(exp.experiment_id):
        client.delete_run(run.info.run_id)
    try:
        client.delete_registered_model(MODEL_NAME)
    except Exception:
        pass
    client.delete_experiment(exp.experiment_id)
    print("Cleanup complete.")
else:
    print("Nothing to clean up.")